# Práctica — Ejercicio 4: Población argentina — interpolación censal

Utilizando los datos del INDEC, completar los años **1978**, **1986** y **2014** mediante
interpolación lineal entre los censos nacionales disponibles.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## Datos censales conocidos

> **Utilizando una interpolación lineal, complete la información sobre Población total para aquellos años en los que no se cuenta con datos de censos nacionales (1978, 1986, 2014).**

In [ ]:
# Datos de censos nacionales conocidos
censos = {
    1960: 20_013_793,
    1970: 23_364_431,
    1980: 27_949_780,
    1991: 32_615_528,
    2001: 36_260_130,
    2010: 40_117_096,
    2022: 46_044_703,
}

# Años a interpolar
anios_faltantes = [1978, 1986, 2014]


def interpolar_censal(anio, datos):
    """Interpolación lineal entre los dos censos que rodean al año dado."""
    anios_censo = sorted(datos.keys())

    if anio <= anios_censo[0]:
        return datos[anios_censo[0]]
    if anio >= anios_censo[-1]:
        return datos[anios_censo[-1]]

    for i in range(len(anios_censo) - 1):
        a0, a1 = anios_censo[i], anios_censo[i + 1]
        if a0 <= anio <= a1:
            t = (anio - a0) / (a1 - a0)
            return round(datos[a0] + t * (datos[a1] - datos[a0]))


print("Interpolación lineal para años sin censo:")
for anio in anios_faltantes:
    pob = interpolar_censal(anio, censos)
    print(f"  {anio}: {pob:,}")

In [ ]:
# Tabla completa con los valores interpolados
tabla = {
    1960: 20_013_793,
    1970: 23_364_431,
    1978: interpolar_censal(1978, censos),
    1980: 27_949_780,
    1986: interpolar_censal(1986, censos),
    1991: 32_615_528,
    2001: 36_260_130,
    2010: 40_117_096,
    2014: interpolar_censal(2014, censos),
    2022: 46_044_703,
}

df_tabla = pd.DataFrame(list(tabla.items()), columns=['Año', 'Población total'])
df_tabla['Dato'] = df_tabla['Año'].apply(
    lambda a: 'Interpolado' if a in anios_faltantes else 'Censo'
)
print(df_tabla.to_string(index=False))

In [ ]:
# Visualización
censos_df = df_tabla[df_tabla['Dato'] == 'Censo']
interpolados_df = df_tabla[df_tabla['Dato'] == 'Interpolado']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_tabla['Año'], df_tabla['Población total'] / 1e6,
        color='steelblue', linewidth=1.5, label='Interpolación lineal')
ax.scatter(censos_df['Año'], censos_df['Población total'] / 1e6,
           color='steelblue', s=80, zorder=5, label='Datos de censo')
ax.scatter(interpolados_df['Año'], interpolados_df['Población total'] / 1e6,
           color='firebrick', marker='D', s=80, zorder=5, label='Valores interpolados')

for _, row in interpolados_df.iterrows():
    ax.annotate(f"{int(row['Población total']/1e6):.1f}M",
                (row['Año'], row['Población total'] / 1e6),
                textcoords='offset points', xytext=(5, 5))

ax.set_xlabel('Año')
ax.set_ylabel('Población (millones)')
ax.set_title('Evolución de la población argentina (INDEC)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones

Valores interpolados linealmente:

| Año | Población interpolada |
|-----|----------------------|
| 1978 | ~27.0 millones (entre 1970 y 1980) |
| 1986 | ~30.5 millones (entre 1980 y 1991) |
| 2014 | ~42.1 millones (entre 2010 y 2022) |

La interpolación lineal asume un crecimiento constante entre censos, lo cual es una aproximación razonable para períodos cortos.